In [ ]:
!pip install datasets==2.19.0 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
from datasets import load_dataset
import pandas as pd

print("Downloading the dataset...")
# 1. Download the German subset
dataset = load_dataset("BioMistral/BioInstructQA", "MedQA", trust_remote_code=True)
df = pd.DataFrame(dataset['test_german'])

# 2. Filter the dataset! We ONLY want MedQA, throw away MedMCQA and the others!
df_medqa = df[df['corpus_name'] == 'MedQA'].copy()

# 3. Save it to a clean CSV file
df_medqa.to_csv("Pure_German_MedQA.csv", index=False)

print(f"Success! Extracted {len(df_medqa)} pure German MedQA questions.")
print("\nHere is the first German question:")
print(df.iloc[0]['prompt_no_answer'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test_french split: 0 examples [00:00, ? examples/s]

...

Generating test_german split: 0 examples [00:00, ? examples/s]

Generating test_portuguese split: 0 examples [00:00, ? examples/s]

Generating test_russian split: 0 examples [00:00, ? examples/s]

Generating test_spanish split: 0 examples [00:00, ? examples/s]

Success! Extracted 1273 pure German MedQA questions.

Here is the first German question:
The following are multiple choice questions (with answers) about medical knowledge. 
 **Question:** Eine 44-jährige afroamerikanische Frau kommt zu ihrem Arzt für eine Routineuntersuchung. Sie macht sich Sorgen um Krebs, weil ihr Onkel vor einem Jahr an metastasierendem Melanom gestorben ist. Sie hat keine Vorgeschichte schwerer Krankheiten und nimmt keine Medikamente ein. Sie arbeitet seit 20 Jahren in einer Anwaltskanzlei und reist regelmäßig mit ihrem Ehemann in die Karibik. Die Untersuchung ihrer Haut zeigt keine abnormen Muttermale oder Warzen. Diese Frau hat das größte Risiko für welchen Typ von Melanom? 
 (A) "Desmoplastisch" 
 (B) "Knötchenförmig" 
 (C) Acral lentiginous 
 (D) "Superficial spreading" - oberflächliche Ausbreitung 
 **Answer:**(


In [ ]:
import pandas as pd

# 1. Load the data we made earlier
df_german = pd.DataFrame(dataset['test_german'])

# 2. Keep ONLY the ID, the Question, and the Solution columns!
clean_df = df_german[['identifier', 'prompt_no_answer', 'prompt']].copy()

# Let's rename the columns so they make perfect sense to you
clean_df.columns = ['ID', 'German_Question', 'Question_With_Correct_Answer']

# 3. Save it!
clean_df.to_csv("Cleaned_German_MedQA.csv", index=False)

print("Successfully cleaned the dataset! All the confusing 'fewshot' columns are gone.")

Successfully cleaned the dataset! All the confusing 'fewshot' columns are gone.


In [ ]:
import pandas as pd

# 1. Load the German test data
df_german = pd.DataFrame(dataset['test_german'])

# 2. MAGIC TRICK: Extract ONLY the last letter from the 'prompt' column
# Since the text ends with "**Answer:**(C", we tell Python to just grab the very last character ([-1])
df_german['Correct_Letter'] = df_german['prompt'].str[-1]

# 3. Create our final, perfectly clean table
clean_df = df_german[['identifier', 'prompt_no_answer', 'Correct_Letter']].copy()

# Rename the columns so they are easy to understand
clean_df.columns = ['ID', 'German_Question', 'Correct_Answer']

# 4. Save it
clean_df.to_csv("Perfect_German_MedQA.csv", index=False)

# Print the first 3 rows to prove it worked!
print(clean_df.head(3))

    ID                                    German_Question Correct_Answer
0   82  The following are multiple choice questions (w...              C
1  465  The following are multiple choice questions (w...              C
2  236  The following are multiple choice questions (w...              A


In [ ]:
from datasets import load_dataset
import pandas as pd

print("Downloading the data...")
dataset = load_dataset("BioMistral/BioInstructQA", "MedQA", trust_remote_code=True)

# 1. Get the GERMAN data and clean it
df_ger = pd.DataFrame(dataset['test_german'])
clean_ger = df_ger[['identifier', 'prompt_no_answer']].copy()
clean_ger.columns = ['ID', 'German_Question']

# 2. Get the ENGLISH data (from the standard 'test' split)
df_eng = pd.DataFrame(dataset['test'])
clean_eng = df_eng[['identifier', 'prompt_no_answer', 'prompt']].copy()

# We grab the answer letter (A, B, C, or D) from the English text
clean_eng['Correct_Answer'] = clean_eng['prompt'].str[-1]
clean_eng = clean_eng[['identifier', 'prompt_no_answer', 'Correct_Answer']]
clean_eng.columns = ['ID', 'English_Question', 'Correct_Answer']

# 3. THE MAGIC STEP: Merge (Join) using the ID!
# This guarantees that ID 82 in English is locked to ID 82 in German, no matter the row order.
merged_df = pd.merge(clean_eng, clean_ger, on='ID', how='inner')

# Reorder the columns so they look nice
merged_df = merged_df[['ID', 'English_Question', 'German_Question', 'Correct_Answer']]

# 4. Save it!
merged_df.to_csv("Bilingual_MedQA_Matched.csv", index=False)

print(f"\nSuccess! {len(merged_df)} questions were perfectly synchronized by their IDs.")
print("\nHere is Row 1 to verify:")
print("ID:", merged_df.iloc[0]['ID'])
print("ENG:", merged_df.iloc[0]['English_Question'][:200], "...")
print("GER:", merged_df.iloc[0]['German_Question'][:200], "...")


Success! 1273 questions were perfectly synchronized by their IDs.

Here is Row 1 to verify:
ID: 82
ENG: The following are multiple choice questions (with answers) about medical knowledge. 
 **Question:** A 44-year-old African-American woman comes to the physician for a routine examination. She is concer ...
GER: The following are multiple choice questions (with answers) about medical knowledge. 
 **Question:** Eine 44-jährige afroamerikanische Frau kommt zu ihrem Arzt für eine Routineuntersuchung. Sie macht s ...


### This dataset is for different medical fields ! We need to filter it and search for status treated by General Practitioner

In [ ]:
import pandas as pd

print("Loading the bilingual benchmark...")
# 1. Load the dataset you just created
df = pd.read_csv("Bilingual_MedQA_Matched.csv")

# 2. Define our GP Keywords (English and German!)
# We use root words so we catch variations (e.g., "depressi" catches depression, depressive, depressiv)
gp_keywords = [
    # 1. Hypertension
    "hypertension", "bluthochdruck", "hypertonie", "blood pressure", "blutdruck",
    # 2. Diabetes
    "diabetes", "diabetic", "diabetiker",
    # 3. Asthma
    "asthma", "asthmatic", "asthmatiker",
    # 4. COPD
    "copd", "obstructive pulmonary", "obstruktive lungen",
    # 5. Coronary Heart Disease
    "coronary", "khk", "koronare", "herzinfarkt", "myocardial infarction", "angina",
    # 6. Heart Failure
    "heart failure", "herzinsuffizienz",
    # 7. Chronic Back Pain
    "back pain", "kreuzschmerz", "rückenschmerz", "lumbar",
    # 8. Depression
    "depression", "depressive", "depressiv",
    # 9. Osteoarthritis
    "osteoarthritis", "arthrose", "joint pain", "gelenkschmerz",
    # 10. CKD & Dementia
    "kidney disease", "nierenkrankheit", "niereninsuffizienz", "renal failure", "ckd",
    "dementia", "demenz", "alzheimer"
]

# 3. Create a massive Regex (Search) Pattern: "word1|word2|word3"
pattern = '|'.join(gp_keywords)

print("Filtering for GP Top 10 Diseases...")
# 4. Apply the filter: Keep the row IF the English OR German question contains a keyword
filtered_df = df[
    df['English_Question'].str.contains(pattern, case=False, na=False) |
    df['German_Question'].str.contains(pattern, case=False, na=False)
].copy()

# 5. Save the final Master's Thesis Benchmark!
filtered_df.to_csv("GP_Top10_Bilingual_Benchmark.csv", index=False)

# Let's see how many questions survived!
print(f"\nOriginal size: {len(df)} questions.")
print(f"New GP-specific size: {len(filtered_df)} questions.")
print("\nSuccess! Saved as 'GP_Top10_Bilingual_Benchmark.csv'.")

Loading the bilingual benchmark...
Filtering for GP Top 10 Diseases...

Original size: 1273 questions.
New GP-specific size: 715 questions.

Success! Saved as 'GP_Top10_Bilingual_Benchmark.csv'.


Modern AI datasets are saved using a universal text format called UTF-8, which perfectly understands German umlauts (ä, ö, ü, ß).

In [ ]:
# 2. Save it with the Windows Excel fix! ('utf-8-sig')
filtered_df.to_csv("GP_Top10_Bilingual_Excel_Fixed.csv", encoding="utf-8-sig", index=False)

In [ ]:
# display the whole data in a row
pd.set_option('display.max_colwidth', None)
filtered_df.head()

,ID,English_Question,German_Question,Correct_Answer
1,465,"The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** A 45-year-old female presents to the emergency department with gross hematuria and acute, colicky flank pain. She denies any previous episodes of hematuria. She reports taking high doses of acetaminophen and aspirin over several weeks due to persistent upper back pain. The patient’s blood pressure and temperature are normal, but she is found to have proteinuria. Physical examination is negative for palpable flank masses. Which of the following is the most likely diagnosis: \n (A) Diffuse cortical necrosis \n (B) Chronic pyelonephritis \n (C) Papillary necrosis \n (D) Acute Nephrolithiasis \n **Answer:**(","The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** ""Eine 45-jährige Frau stellt sich in der Notaufnahme mit sichtbarem Blut im Urin und akuten, kolikartigen Flankenschmerzen vor. Sie verneint frühere Episoden von Hämaturie. Sie berichtet, dass sie aufgrund anhaltender Schmerzen im oberen Rücken über mehrere Wochen hohe Dosen Acetaminophen und Aspirin eingenommen hat. Der Blutdruck und die Temperatur der Patientin sind normal, aber es wird eine Proteinurie festgestellt. Die körperliche Untersuchung ergibt keinen tastbaren Flankentumor. Welche der folgenden Diagnosen ist am wahrscheinlichsten:"" \n (A) ""Difuse kortikale Nekrose"" \n (B) ""Chronische Pyelonephritis"" \n (C) ""Papilläre Nekrose"" \n (D) ""Akute Nephrolithiasis"" \n **Answer:**(",C
3,481,"The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** Current recommendations state that a single hemoglobin A1c value of greater than 6.5% is diagnostic of diabetes mellitus. If this 6.5% cut-off is to be increased to 7.0%, which of the following would be true? \n (A) Increase in false negative test results \n (B) Increase in false positive test results \n (C) Decrease in true negative test results \n (D) Increase in true positive test results \n **Answer:**(","The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** ""Nach aktuellen Empfehlungen bedeutet ein einzelner Hämoglobin A1c-Wert von über 6,5% eine Diagnose von Diabetes mellitus. Wenn dieser Grenzwert von 6,5% auf 7,0% erhöht werden soll, welches der folgenden Aussagen wäre wahr?"" \n (A) ""Zunahme falsch negativer Testergebnisse"" \n (B) ""Zunahme von falsch positiven Testergebnissen"" \n (C) ""Abnahme der wahren negativen Testergebnisse"" \n (D) ""Anstieg der wahren positiven Testergebnisse"" \n **Answer:**(",A
11,585,"The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** A 55-year-old man is brought to the emergency department 12 hours after the sudden onset of shortness of breath and substernal chest pain at rest; the pain is increased by inspiration. He has also had a nonproductive cough, fever, and malaise for the past 5 days. He does not smoke or use illicit drugs. His temperature is 38°C (100.4°F), pulse is 125/min, respirations are 32/min, and blood pressure is 85/45 mm Hg. Physical examination shows distended neck veins. Auscultation of the chest discloses bilateral basilar rales and muffled heart sounds. An ECG shows sinus tachycardia, diffuse ST segment elevation, low voltage QRS complexes, and fluctuating R wave amplitude. Which of the following is the most likely diagnosis? \n (A) Kawasaki disease \n (B) Rheumatic fever \n (C) Infective endocarditis \n (D) Cardiac tamponade \n **Answer:**(","The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** Ein 55-jähriger Mann wird 12 Stunden nach dem plötzlichen Auftreten von Atemnot und substernalem Brustschmerz in die Notaufnahme gebracht; der Schmerz wird durch Einatmen verstärkt. Er hatte in den letzten 5 Tagen auch einen nicht produktiven Husten, Fieber und Abgeschlagenhe

Because our Benschmark contains the Multiple choice answers , and we need to know how the LLM thinks without giving it answers , we add the questions without the MCQ answers to the dataset. Here is the final result !

In [ ]:
import pandas as pd
# Load the dataset
df_openQ = pd.read_csv('/content/GP_Top10_Bilingual_Open_Ended.csv')

In [ ]:
# DROP THE EMPTY COLUMN
df_openQ.drop(columns=['Unnamed: 6'], inplace=False)


,ID,English_Question,German_Question,Correct_Answer,English_Open_Question,German_Open_Question
0,465,"The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** A 45-year-old female presents to the emergency department with gross hematuria and acute, colicky flank pain. She denies any previous episodes of hematuria. She reports taking high doses of acetaminophen and aspirin over several weeks due to persistent upper back pain. The patient’s blood pressure and temperature are normal, but she is found to have proteinuria. Physical examination is negative for palpable flank masses. Which of the following is the most likely diagnosis: \n (A) Diffuse cortical necrosis \n (B) Chronic pyelonephritis \n (C) Papillary necrosis \n (D) Acute Nephrolithiasis \n **Answer:**(","The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** ""Eine 45-jährige Frau stellt sich in der Notaufnahme mit sichtbarem Blut im Urin und akuten, kolikartigen Flankenschmerzen vor. Sie verneint frühere Episoden von Hämaturie. Sie berichtet, dass sie aufgrund anhaltender Schmerzen im oberen Rücken über mehrere Wochen hohe Dosen Acetaminophen und Aspirin eingenommen hat. Der Blutdruck und die Temperatur der Patientin sind normal, aber es wird eine Proteinurie festgestellt. Die körperliche Untersuchung ergibt keinen tastbaren Flankentumor. Welche der folgenden Diagnosen ist am wahrscheinlichsten:"" \n (A) ""Difuse kortikale Nekrose"" \n (B) ""Chronische Pyelonephritis"" \n (C) ""Papilläre Nekrose"" \n (D) ""Akute Nephrolithiasis"" \n **Answer:**(",C,"A 45-year-old female presents to the emergency department with gross hematuria and acute, colicky flank pain. She denies any previous episodes of hematuria. She reports taking high doses of acetaminophen and aspirin over several weeks due to persistent upper back pain. The patient’s blood pressure and temperature are normal, but she is found to have proteinuria. Physical examination is negative for palpable flank masses. What is the most likely diagnosis:","Eine 45-jährige Frau stellt sich in der Notaufnahme mit sichtbarem Blut im Urin und akuten, kolikartigen Flankenschmerzen vor. Sie verneint frühere Episoden von Hämaturie. Sie berichtet, dass sie aufgrund anhaltender Schmerzen im oberen Rücken über mehrere Wochen hohe Dosen Acetaminophen und Aspirin eingenommen hat. Der Blutdruck und die Temperatur der Patientin sind normal, aber es wird eine Proteinurie festgestellt. Die körperliche Untersuchung ergibt keinen tastbaren Flankentumor. Welche Diagnose ist am wahrscheinlichsten:"
1,481,"The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** Current recommendations state that a single hemoglobin A1c value of greater than 6.5% is diagnostic of diabetes mellitus. If this 6.5% cut-off is to be increased to 7.0%, which of the following would be true? \n (A) Increase in false negative test results \n (B) Increase in false positive test results \n (C) Decrease in true negative test results \n (D) Increase in true positive test results \n **Answer:**(","The following are multiple choice questions (with answers) about medical knowledge. \n **Question:** ""Nach aktuellen Empfehlungen bedeutet ein einzelner Hämoglobin A1c-Wert von über 6,5% eine Diagnose von Diabetes mellitus. Wenn dieser Grenzwert von 6,5% auf 7,0% erhöht werden soll, welches der folgenden Aussagen wäre wahr?"" \n (A) ""Zunahme falsch negativer Testergebnisse"" \n (B) ""Zunahme von falsch positiven Testergebnissen"" \n (C) ""Abnahme der wahren negativen Testergebnisse"" \n (D) ""Anstieg der wahren positiven Testergebnisse"" \n **Answer:**(",A,"Current recommendations state that a single hemoglobin A1c value of greater than 6.5% is diagnostic of diabetes mellitus. If this 6.5% cut-off is to be increased to 7.0%, What would be true?","Nach aktuellen Empfehlungen bedeutet ein einzelner Hämoglobin A1c-Wert von über 6,5% eine Diagnose vo

In [ ]:
df_openQ.to_csv("GP_Top10_Bilingual_Open_EndedQ.csv", encoding="utf-8-sig", index=False)

In [ ]:
from google.colab import files
files.download('GP_Top10_Bilingual_Open_EndedQ.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Our Benschmark is finally ready :)